# Figure S6 2014,2022 manual clusters

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import os
import time

import plotly.offline as pyo
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white" # https://medium.com/plotly/introducing-plotly-py-theming-b644109ac9c7

directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\Final datasets used in article'
os.chdir(directory)

#for fig rendering errors:
#plotly.offline.init_notebook_mode(connected=True)
#pio.renderers.default = 'iframe_connected' 
#from plotly.offline import init_notebook_mode, iplot
#init_notebook_mode(connected=True)
#pyo.init_notebook_mode()


In [ ]:
#load data
DF = pd.read_csv(r'28languagesUA.csv', sep=',') #, index_col=[0,1], skipinitialspace=True)
DF.date=pd.to_datetime(DF.date) #,dayfirst=True
DF.sample(3)

In [ ]:
#DF2014 = DF[(DF['date'] >= '2014-01-01') & (DF['date'] <= '2014-04-30')]
#DF2022 = DF[(DF['date'] >= '2022-01-01') & (DF['date'] <= '2022-04-30')]

#DF2014 = DF[(DF['date'] >= '2014-01-30') & (DF['date'] <= '2014-03-27')] #Enne valitud
#DF2022 = DF[(DF['date'] >= '2022-01-27') & (DF['date'] <= '2022-03-24')] # Enne valitud

#DF2014 = DF[(DF['date'] >= '2014-02-06') & (DF['date'] <= '2014-03-19')]
#DF2022 = DF[(DF['date'] >= '2022-02-03') & (DF['date'] <= '2022-03-16')]



# Define the central dates as datetime objects
central_dates = { 
    '2014': pd.to_datetime('2014-02-18'),
    '2022': pd.to_datetime('2022-02-24')
}

# Filter the DataFrame based on 4 weeks before and after the central dates
#DF2014 = DF[(DF['date'] >= central_dates['2014'] - pd.Timedelta(weeks=1)) & 
#            (DF['date'] <= central_dates['2014'] + pd.Timedelta(weeks=3))]
DF2014 = DF[(DF['date'] >= central_dates['2014'] - pd.Timedelta(weeks=4)) & 
            (DF['date'] <= central_dates['2014'] + pd.Timedelta(weeks=4))]

DF2022 = DF[(DF['date'] >= central_dates['2022'] - pd.Timedelta(weeks=4)) & 
            (DF['date'] <= central_dates['2022'] + pd.Timedelta(weeks=4))]

DF2014.head(3)

DF2014['date'] = DF2014['date'] + pd.DateOffset(years=8, days=6) #offset 2014 to make em comparable

DF2014.head(3)

##### Normalized

In [ ]:
#NORMALIZE BY LOCAL AVG
import numpy as np
import pandas as pd
Overal_mean_freq = DF.groupby('language')['freq'].transform('mean')

def calculate_normalized_frequency(df, freq_col='freq', lang_col='language'):
    mean_freq = df.groupby(lang_col)[freq_col].transform('mean')
    normalized_freq = (df[freq_col] / mean_freq)
    return normalized_freq

def calculate_normalized_overal_frequency(df, freq_col='freq', lang_col='language'):
    normalized_freq = np.log10(df[freq_col] / Overal_mean_freq)
    return normalized_freq

# Calculate overall mean frequency for each language in DF
Overal_mean_freq = DF.groupby('language')['freq'].mean()

def calculate_normalized_overal_frequency(df, overall_mean_freq, freq_col='freq', lang_col='language'):
    # Ensure we only calculate for valid entries
    normalized_freq = (df[freq_col] / overall_mean_freq[df[lang_col]])
    return normalized_freq


# Example usage:
DF2014['normalized_loc_freq'] = calculate_normalized_frequency(DF2014)
DF2022['normalized_loc_freq'] = calculate_normalized_frequency(DF2022)
DF2014['normalized_overal_freq'] = calculate_normalized_frequency(DF2014)
DF2022['normalized_overal_freq'] = calculate_normalized_frequency(DF2022)
DF2014['normalized_overal_freq_log'] = np.log10(calculate_normalized_frequency(DF2014))
DF2022['normalized_overal_freq_log'] = np.log10(calculate_normalized_frequency(DF2022))
DF2022['freq_log'] = np.log10(DF2022['freq'])
DF2014['freq_log'] = np.log10(DF2014['freq'])


DF2022.head(3)

In [ ]:
lang2cluster = {
    # Cluster 1: persistently high attention
    "Ukrainian": "Stable high attention",
    "Russian":   "Stable high attention",

    # Cluster 2: short-lived, transcontinental spikes
    "Turkish":    "Short-lived reaction",
    "Portuguese": "Short-lived reaction",
    "Arabic":     "Short-lived reaction",
    "Catalan":    "Short-lived reaction",
    "Korean":     "Short-lived reaction",
    "Persian":    "Short-lived reaction",

    # Cluster 3: strong 2014 spike only
    "Estonian":  "2014-sustained only",
    "Serbian":   "2014-sustained only",
    "Hungarian": "2014-sustained only",
    "Vietnamese":"2014-sustained only",
    "Urdu":      "2014-sustained only",

    # Cluster 4: muted in 2014 → strong, prolonged 2022
    "German":     "EU-heavy 2022 reaction",
    "Italian":    "EU-heavy 2022 reaction",
    "French":     "EU-heavy 2022 reaction",
    "Finnish":    "EU-heavy 2022 reaction",
    "Norwegian":  "EU-heavy 2022 reaction",
    "Dutch":      "EU-heavy 2022 reaction",
    "English":    "EU-heavy 2022 reaction",
    "Danish":     "EU-heavy 2022 reaction",
    "Czech":      "EU-heavy 2022 reaction",
    "Swedish":    "EU-heavy 2022 reaction",
    "Polish":     "EU-heavy 2022 reaction",

    # Cluster 5: geostrategic neighbours with sustained attention
    "Romanian": "Geostrategic sustained attention",
    "Greek":    "Geostrategic sustained attention",

    # Explicit outliers
    #"Spanish": "Outlier: stronger 2014 attention",
}


In [ ]:

# 1. Tag each row with its cluster and year
DF2014['cluster'] = DF2014['language'].map(lang2cluster)
DF2022['cluster'] = DF2022['language'].map(lang2cluster)
DF2014['year']    = '2014'
DF2022['year']    = '2022'

# 2. Combine into one DataFrame
df = pd.concat([DF2014, DF2022], ignore_index=True)


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import os
from subprocess import call

# ============================================================
# 1. PREPARE DATA
# ============================================================

df['freq'] = df['freq'].replace(0, np.nan)

cluster_ts = (
    df
    .groupby(['cluster','year','date'])['freq']
    .agg(
      mean='mean',
      q10=lambda x: x.quantile(0.10),
      q90=lambda x: x.quantile(0.90)
    )
    .reset_index()
)

clusters = cluster_ts['cluster'].unique()

fig = make_subplots(
    rows=len(clusters), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02
)

# Y‑axis limits
mean_vals = cluster_ts['mean']
y_min = mean_vals[mean_vals > 0].min()
y_max = mean_vals.max()

log_y_min = np.log10(y_min) - 0.95
log_y_max = np.log10(y_max) + 0.3

# ============================================================
# 2. DEFINE DATE TICKS (NUMERIC FOR KALEIDO)
# ============================================================

feb_24_2022 = pd.to_datetime('2022-02-24')
date_intervals = (
    [feb_24_2022 - pd.Timedelta(weeks=i) for i in range(4, 0, -1)]
    + [feb_24_2022]
    + [feb_24_2022 + pd.Timedelta(weeks=i) for i in range(1, 5)]
)
date_labels = ["-4w","-3w","-2w","-1w","Invasion","+1w","+2w","+3w","+4w"]

# Convert datetime → numeric (milliseconds since epoch)
date_interval_nums = [d.value // 10**6 for d in date_intervals]

# ============================================================
# 3. BUILD FIGURE
# ============================================================

for i, cl in enumerate(clusters, start=1):

    for year, color in [('2014','#4d99c6'), ('2022','#bd2f36')]:

        langs = df.loc[(df['cluster']==cl) & (df['year']==year), 'language'].unique()
        for lang in langs:
            dlang = df.query("cluster==@cl and year==@year and language==@lang")

            # Convert x to numeric
            x_numeric = dlang['date'].astype('int64') // 10**6

            fig.add_trace(
                go.Scatter(
                    x=x_numeric,
                    y=dlang['freq'],
                    mode='lines',
                    line=dict(color=color, width=2),
                    opacity=0.2,
                    connectgaps=False,
                    showlegend=False
                ),
                row=i, col=1
            )

        dmean = cluster_ts.query("cluster==@cl and year==@year")
        x_mean_numeric = dmean['date'].astype('int64') // 10**6

        fig.add_trace(
            go.Scatter(
                x=x_mean_numeric,
                y=dmean['mean'],
                mode='lines',
                line=dict(color=color, width=3),
                connectgaps=False,
                showlegend=False
            ),
            row=i, col=1
        )

    # X‑axis (numeric ticks)
    fig.update_xaxes(
        row=i, col=1,
        tickmode='array',
        tickvals=date_interval_nums,
        ticktext=date_labels,
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        tickfont=dict(size=26),
        title_font=dict(size=28),
        type='linear'   # <<< IMPORTANT
    )

    # Y‑axis
    fig.update_yaxes(
        row=i, col=1,
        title_text=cl,
        type='log',
        range=[log_y_min, log_y_max],
        tickfont=dict(size=26),
        title_font=dict(size=36)
    )

# ============================================================
# 4. LAYOUT
# ============================================================

fig.update_layout(
    width=800,
    height=300 * len(clusters),
    showlegend=False,
    margin=dict(l=150, r=30, t=30, b=70),
    title="",
    font=dict(size=26, family="Arial")
)

fig.update_xaxes(fixedrange=True)
fig.update_yaxes(fixedrange=True)

# ============================================================
# 5. SAVE FIGURE — DO NOT TOUCH AXES HERE
# ============================================================

fig.full_figure_for_development(warn=False)

# Paths
sublocation = '/Visualizations/Fig.S6_5 categories trends/trendclusters/'
name = 'trendclusters'

directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter'
os.chdir(directory)

main_figure_path = directory + sublocation
os.makedirs(main_figure_path, exist_ok=True)

formats = ['pdf', 'png', 'svg']
for fmt in formats:
    os.makedirs(main_figure_path + fmt + "/", exist_ok=True)

# Export
for fmt in formats:
    file_path = main_figure_path + fmt + "/" + name + "." + fmt
    fig.write_image(
        file_path,
        format=fmt,
        engine="kaleido",
        width=1600,
        height=600 * len(clusters),
        scale=2
    )
    print(f"Saved {fmt.upper()} to: {file_path}")
